In [27]:
import numpy as np
import pandas as pd
import xarray as xr
import os
import glob

In [28]:
# pick only the daily files you want
tas_files  = sorted(glob.glob("w5e5daily/tas_W5E5v2.0_*.nc"))
pr_files   = sorted(glob.glob("w5e5daily/pr_W5E5v2.0_*.nc"))
rsds_files = sorted(glob.glob("w5e5daily/rsds_W5E5v2.0_*.nc"))

# Annual

In [ ]:
# good default chunking for big global grids; tweak to your HPC + analysis pattern
chunks = {"time": 366, "lat": 360, "lon": 720}
def open_var(files):
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=True,
        chunks=chunks,
        data_vars="minimal",
        coords="minimal", 
        engine=None, 
        compat="override",
    )
tas  = open_var(tas_files)
pr   = open_var(pr_files)
rsds = open_var(rsds_files)
tas_da=tas['tas']
rsds_da=rsds['rsds']
pr_da=pr['pr']

In [7]:
def annual_stats(da):
    g = da.resample(time="YS")
    return xr.Dataset({
        "mean":   g.mean("time"),
        "min":    g.min("time"),
        "max":    g.max("time"),
        "median": g.median("time"),
        "std":    g.std("time"),
    })

tas_m  = annual_stats(tas_da).rename({k: f"tas_{k}"  for k in ["mean","min","max","median","std"]})
rsds_m = annual_stats(rsds_da).rename({k: f"rsds_{k}" for k in ["mean","min","max","median","std"]})

In [21]:
# pr: kg m-2 s-1 -> mm/day then sum to mm/month
pr_mm_month = (pr_da * 86400.0).resample(time="YS").sum("time").rename("pr_sum").to_dataset()
pr_mm_month["pr_sum"].attrs["units"] = "mm month-1"

monthly = xr.merge([tas_m, rsds_m, pr_mm_month], compat="override")

In [22]:
monthly.to_netcdf("w5e5aggregrates/W5E5_calyr_aggregates.nc")

## Hydrological Year

In [29]:
# good default chunking for big global grids; tweak to your HPC + analysis pattern
chunks = {"time": 366, "lat": 360, "lon": 720}
def open_var(files):
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=True,
        chunks=chunks,
        data_vars="minimal",
        coords="minimal", 
        engine=None, 
        compat="override",
    )
tas  = open_var(tas_files)
pr   = open_var(pr_files)
rsds = open_var(rsds_files)
tas_da=tas['tas']
rsds_da=rsds['rsds']
pr_da=pr['pr']

In [32]:
def annual_stats(da):
    g = da.resample(time="YS-OCT")
    return xr.Dataset({
        "mean":   g.mean("time"),
        "min":    g.min("time"),
        "max":    g.max("time"),
        "median": g.median("time"),
        "std":    g.std("time"),
    })

tas_m  = annual_stats(tas_da).rename({k: f"tas_{k}"  for k in ["mean","min","max","median","std"]})
rsds_m = annual_stats(rsds_da).rename({k: f"rsds_{k}" for k in ["mean","min","max","median","std"]})

In [33]:
# pr: kg m-2 s-1 -> mm/day then sum to mm/month
pr_mm_month = (pr_da * 86400.0).resample(time="YS-OCT").sum("time").rename("pr_sum").to_dataset()
pr_mm_month["pr_sum"].attrs["units"] = "mm month-1"

monthly = xr.merge([tas_m, rsds_m, pr_mm_month], compat="override")

In [34]:
monthly.to_netcdf("w5e5aggregrates/W5E5_nhhydyr_aggregates.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/lus/lfs1aip2/scrat

In [35]:
def annual_stats(da):
    g = da.resample(time="YS-APR")
    return xr.Dataset({
        "mean":   g.mean("time"),
        "min":    g.min("time"),
        "max":    g.max("time"),
        "median": g.median("time"),
        "std":    g.std("time"),
    })

tas_m  = annual_stats(tas_da).rename({k: f"tas_{k}"  for k in ["mean","min","max","median","std"]})
rsds_m = annual_stats(rsds_da).rename({k: f"rsds_{k}" for k in ["mean","min","max","median","std"]})
# pr: kg m-2 s-1 -> mm/day then sum to mm/month
pr_mm_month = (pr_da * 86400.0).resample(time="YS-APR").sum("time").rename("pr_sum").to_dataset()
pr_mm_month["pr_sum"].attrs["units"] = "mm month-1"

monthly = xr.merge([tas_m, rsds_m, pr_mm_month], compat="override")
monthly.to_netcdf("w5e5aggregrates/W5E5_shhydyr_aggregates.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/lus/lfs1aip2/scrat

# Seasonal

# Monthly

In [5]:
# good default chunking for big global grids; tweak to your HPC + analysis pattern
chunks = {"time": 31, "lat": 360, "lon": 720}

In [6]:
def open_var(files):
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=True,
        chunks=chunks,
        data_vars="minimal",
        coords="minimal", 
        engine=None, 
        compat="override",
    )

In [7]:
tas  = open_var(tas_files)
pr   = open_var(pr_files)
rsds = open_var(rsds_files)

In [8]:
tas

<xarray.Dataset> Size: 16GB
Dimensions:  (time: 14975, lat: 360, lon: 720)
Coordinates:
  * time     (time) datetime64[ns] 120kB 1979-01-01 1979-01-02 ... 2019-12-31
  * lat      (lat) float64 3kB 89.75 89.25 88.75 88.25 ... -88.75 -89.25 -89.75
  * lon      (lon) float64 6kB -179.8 -179.2 -178.8 -178.2 ... 178.8 179.2 179.8
Data variables:
    tas      (time, lat, lon) float32 16GB dask.array<chunksize=(31, 360, 720), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7
    title:        WFDE5 over land merged with ERA5 over the ocean (W5E5) vers...
    institution:  Potsdam Institute for Climate Impact Research (PIK)
    project:      Inter-Sectoral Impact Model Intercomparison Project phase 3...
    contact:      ISIMIP cross-sectoral science team <info@isimip.org> <https...
    summary:      WFDE5 (with GPCC precipitation correction) over land merged...
    references:   Cucchi et al. (2020) <https://doi.org/10.5194/essd-12-2097-...
    version:      2.0

In [8]:
tas['tas']

<xarray.DataArray 'tas' (time: 14975, lat: 360, lon: 720)> Size: 16GB
dask.array<concatenate, shape=(14975, 360, 720), dtype=float32, chunksize=(31, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 120kB 1979-01-01 1979-01-02 ... 2019-12-31
  * lat      (lat) float64 3kB 89.75 89.25 88.75 88.25 ... -88.75 -89.25 -89.75
  * lon      (lon) float64 6kB -179.8 -179.2 -178.8 -178.2 ... 178.8 179.2 179.8
Attributes:
    standard_name:  air_temperature
    long_name:      Near-Surface Air Temperature
    units:          K

In [10]:
tas_da=tas['tas']
rsds_da=rsds['rsds']
pr_da=pr['pr']

In [12]:
def monthly_stats(da):
    g = da.resample(time="MS")
    return xr.Dataset({
        "mean":   g.mean("time"),
        "min":    g.min("time"),
        "max":    g.max("time"),
        "median": g.median("time"),
        "std":    g.std("time"),
    })

tas_m  = monthly_stats(tas_da).rename({k: f"tas_{k}"  for k in ["mean","min","max","median","std"]})
rsds_m = monthly_stats(rsds_da).rename({k: f"rsds_{k}" for k in ["mean","min","max","median","std"]})

In [13]:
# pr: kg m-2 s-1 -> mm/day then sum to mm/month
pr_mm_month = (pr_da * 86400.0).resample(time="MS").sum("time").rename("pr_monthly_total").to_dataset()
pr_mm_month["pr_monthly_total"].attrs["units"] = "mm month-1"

monthly = xr.merge([tas_m, rsds_m, pr_mm_month], compat="override")

monthly.to_netcdf("w5e5monthly/W5E5_monthly_aggregates.nc")

HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 496 in H5Fis_accessible(): unable to determine if file is accessible as HDF5
    major: File accessibility
    minor: Not an HDF5 file
  #001: H5VLcallback.c line 3913 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #002: H5VLcallback.c line 3848 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLnative_file.c line 344 in H5VL__native_file_specific(): error in HDF5 file check
    major: File accessibility
    minor: Can't get value
  #004: H5Fint.c line 1055 in H5F__is_hdf5(): unable to open file
    major: File accessibility
    minor: Unable to initialize object
  #005: H5FD.c line 787 in H5FD_open(): can't open file
    major: Virtual File Layer
    minor: Unable to open file
  #006: H5FDsec2.c line 323 in H5FD__sec2_open(): unable to open file: name = '/lus/lfs1aip2/scrat